# 04 · Baselines on validation, and the MPC-CC risk level

All baseline controllers on the 73 validation weeks (Mathura, Bareilly, Uttar Pradesh). The chance-constrained MPC's risk level
ε ∈ {0.20, 0.10, 0.05} is chosen here (best mean validation reward) and frozen for the test.

In [1]:
import sys, os, json, hashlib, pickle, time, warnings
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); os.environ["PYTHONPATH"] = str(ROOT / "src")
PROC, RES = ROOT / "data" / "processed", ROOT / "results"
assert hashlib.sha256((RES / "FINAL_SPEC.md").read_bytes()).hexdigest() == json.loads((RES / "FINAL_SPEC_lock.json").read_text())["sha256"], "plan changed"
from sim import load_cfg, load_bank
CFG = load_cfg(ROOT); BANK_PATH = str(PROC / "bank_trainval.pkl")
VAL = json.loads((PROC / "episodes.json").read_text())["val"]
EPS_CC = [0.20, 0.10, 0.05]

def job(kind, ids, bank_path, cfg, val, arg=None):
    import sys, os; sys.path.insert(0, os.environ["PYTHONPATH"])
    from dataclasses import replace
    from sim import load_bank
    from controllers import run_policy, run_perfect, run_mpc
    bank = load_bank(bank_path); out = []
    for i in ids:
        if kind == "none": r = run_policy(bank, val, replace(cfg, pv_kwp_per_home=0.0, batt_kwh_per_home=0.0, batt_kw_per_home=0.0), "idle_full", i)
        elif kind == "solar": r = run_policy(bank, val, replace(cfg, batt_kwh_per_home=0.0, batt_kw_per_home=0.0), "idle_full", i)
        elif kind == "backup": r = run_policy(bank, val, cfg, "backup_only", i)
        elif kind == "rule": r = run_policy(bank, val, cfg, "rule", i)
        elif kind == "rule_shield": r = run_policy(bank, val, cfg, "rule", i, shield=True)
        elif kind == "pf": r = run_perfect(bank, val, cfg, i, time_limit=600)
        elif kind == "mpc_v2": r = run_mpc(bank, val, cfg, i, "v2")
        else: r = run_mpc(bank, val, cfg, i, "cc", eps_cc=arg)
        out.append(r)
    return out

NAMES = {"none": "0 No system", "solar": "1 Solar only", "backup": "2 Backup-only inverter", "rule": "3 Rule-based",
         "rule_shield": "3s Rule-based + shield", "mpc_v2": "4a MPC v2 (risk-aware)", "pf": "5 Perfect foresight"}
JOBS = [(k, None) for k in NAMES] + [("mpc_cc", e) for e in EPS_CC]
chunks = [list(range(i, len(VAL), 16)) for i in range(16)]
t0 = time.time(); rows = []
for kind, arg in JOBS:
    f = RES / "_val_cache" / f"{kind}_{arg}.pkl"; f.parent.mkdir(exist_ok=True)
    if f.exists(): out = pickle.loads(f.read_bytes())
    else:
        out = [r for part in Parallel(n_jobs=8)(delayed(job)(kind, c, BANK_PATH, CFG, VAL, arg) for c in chunks) for r in part]
        f.write_bytes(pickle.dumps(out))
    name = NAMES[kind] if kind in NAMES else f"4b MPC-CC (eps {arg:.2f})"
    rows += [{"method": name, **r, "region": VAL[r["episode"]]["region"]} for r in out]
    print(f"{name}: done ({time.time() - t0:.0f}s)", flush=True)
VB = pd.DataFrame(rows); VB.to_csv(RES / "val_baselines_per_episode.csv", index=False)

0 No system: done (0s)


1 Solar only: done (8s)


2 Backup-only inverter: done (11s)


3 Rule-based: done (15s)


3s Rule-based + shield: done (19s)


4a MPC v2 (risk-aware): done (1575s)


5 Perfect foresight: done (1661s)


4b MPC-CC (eps 0.20): done (3260s)


4b MPC-CC (eps 0.10): done (4879s)


4b MPC-CC (eps 0.05): done (6508s)


In [2]:
COLS = {"LOLH_ess_h_per_home_month": "Essentials off (h/home·month)", "cut_demand_served_pct": "Cut demand served (%)",
        "evening_import_kwh_per_day": "Evening grid (kWh/day)", "evening_excess_kwh_per_day": "Above cap (kWh/day)",
        "battery_cycles_per_day": "Battery cycles/day", "jain_fairness": "Fairness (Jain)", "shield_rate_pct": "Shield changed (%)",
        "reward_per_week": "Reward/week"}
tab = VB.groupby("method")[list(COLS)].mean().rename(columns=COLS)
display(tab.round(3))
display(VB.pivot_table(index="method", columns="region", values="reward_per_week").round(2))
cc = {e: float(VB[VB.method == f"4b MPC-CC (eps {e:.2f})"].reward_per_week.mean()) for e in EPS_CC}
eps_star = max(cc, key=cc.get)
(RES / "mpc_cc_choice.json").write_text(json.dumps({"validation_reward_per_week": cc, "eps_star": eps_star,
                                                    "rule": "best mean validation reward (FINAL_SPEC section 6)"}, indent=1))
print("MPC-CC risk level chosen on validation: eps* =", eps_star, cc)
pf = VB[VB.method.str.startswith("5")]; print(f"perfect foresight: {int(pf.pf_time_limited.sum())} of {len(pf)} weeks hit the 600 s cap, largest gap {pf.pf_mip_gap.max():.4f}")
tab.round(3).to_csv(RES / "val_baselines.csv", encoding="utf-8")

,Essentials off (h/home·month),Cut demand served (%),Evening grid (kWh/day),Above cap (kWh/day),Battery cycles/day,Fairness (Jain),Shield changed (%),Reward/week
method,,,,,,,,
0 No system,53.378,0.000,14.618,0.892,0.000,NaN,0.000,-38.549
1 Solar only,22.355,50.102,14.589,0.884,0.000,0.845,0.000,-25.383
2 Backup-only inverter,1.952,95.119,15.852,1.577,0.454,0.993,0.000,-16.843
3 Rule-based,3.321,88.461,12.264,0.819,0.680,0.978,0.000,-15.113
3s Rule-based + shield,2.272,88.511,12.269,0.819,0.679,0.979,0.813,-14.787
4a MPC v2 (risk-aware),4.463,72.798,11.419,0.594,0.693,0.947,10.345,-16.727
4b MPC-CC (eps 0.05),4.422,79.431,11.460,0.567,0.706,0.955,0.000,-15.473
4b MPC-CC (eps 0.10),4.459,79.556,11.448,0.575,0.709,0.955,0.000,-15.469
4b MPC-CC (eps 0.20),4.708,79.631,11.469,0.577,0.708,0.955,0.000,-15.514


region,Bareilly,Mathura,UP
method,,,
0 No system,-48.06,-32.36,-15.17
1 Solar only,-31.65,-20.77,-10.68
2 Backup-only inverter,-21.66,-12.58,-6.50
3 Rule-based,-20.05,-10.57,-4.72
3s Rule-based + shield,-19.49,-10.57,-4.72
4a MPC v2 (risk-aware),-21.90,-12.25,-5.49
4b MPC-CC (eps 0.05),-20.14,-11.45,-5.28
4b MPC-CC (eps 0.10),-20.16,-11.45,-5.21
4b MPC-CC (eps 0.20),-20.23,-11.44,-5.25


MPC-CC risk level chosen on validation: eps* = 0.1 {0.2: -15.513666689488636, 0.1: -15.469043930649129, 0.05: -15.472634943661387}
perfect foresight: 0 of 73 weeks hit the 600 s cap, largest gap 0.0001
